In [ ]:
import geopandas as gpd


#Here, we find the top 5 business districts using land use data and find all stops in ot within 100m of of the commercial land.

#Load data 
quarters   = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\Lux_quaters.geojson").to_crs(epsg=2169)
commercial = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\Commerical_land.gpkg").to_crs(epsg=2169)
stops      = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\stop_freq_avl.gpkg").to_crs(epsg=2169)

#Find the commercial area per quarter
commercial_intersect = gpd.overlay(quarters, commercial, how="intersection")
commercial_intersect["comm_area"] = commercial_intersect.area

#Create a DataFrame summing the commercial area per quarter and add a field for the total quarter area
comm_area_quarter = commercial_intersect.groupby("FK_QUART_NAME")["comm_area"].sum().reset_index()
quarters["quarter_area"] = quarters.geometry.area

#Calculate coverage ratio: comm_area / quarter_area
quarters = quarters.merge(comm_area_quarter, on="FK_QUART_NAME", how="left").fillna(0)
quarters["coverage"] = quarters["comm_area"] / quarters["quarter_area"]

#Select the top five coverage ratios and keep only the commercial areas in those top five districts
top5 = quarters.sort_values("coverage", ascending=False).head(5)
top_comm = gpd.overlay(commercial, top5, how="intersection")
top_comm = top_comm[(top_comm.geometry.area > 1000)]

#Find candidate stops for each district within 100m or in the commercial land itself.
#These stops will be the destinations of the analysis.
top_comm["geometry"] = top_comm.geometry.buffer(100)
stops_near = gpd.sjoin(stops, top_comm, how="inner", predicate="within")
stops_near = stops_near[["FK_QUART_NAME", "stop_id", "geometry"]].drop_duplicates(subset=["FK_QUART_NAME", "stop_id"])
stops_near["node"] = "pt_" + stops_near["stop_id"].astype(str)

#Finally, we keep the top five districts for mapping while the stops_near is for the analysis
top_districts = top5[["FK_QUART_NAME", "coverage", "geometry"]].to_crs(epsg=4326)

top_districts.to_file("business_districts.gpkg", driver="GPKG")
stops_near.to_csv("district_candidate_stops.csv", index=False)


print("Top 5 business districts found and stops assigned.")

Top 5 business districts found and stops assigned.
